<a href="https://colab.research.google.com/github/Jaywestty/finetuned-bart-summarizer/blob/main/Text_summarizer_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **NEWS TEXT SUMMARIZER PROJECT (IMPROVED)**

#### **Project Description**
This project fine-tunes `facebook/bart-base` to summarize news articles into short, factual highlights. The model is trained on a subset of CNN/DailyMail so it fits the Google Colab free tier (T4 GPU).

#### **What changed from the first version**
1. **Evaluation generation fixed.** `generation_max_length=142` and `generation_num_beams` are now set, so the model is no longer cut off at the default of about 20 tokens during evaluation.
2. **Proper ROUGE-Lsum.** Predictions and references are split into sentences before scoring. Without this, `rougeLsum` collapses into `rougeL`.
3. **Reproducible checkpoint selection.** `load_best_model_at_end=True` with `metric_for_best_model="rougeLsum"` replaces the manual choice of epoch 2.
4. **Test split used once.** The test set is only touched at the end, after the best checkpoint is selected.
5. **Lead-3 baseline.** The first three sentences of each article are scored with the same metric so the model has something to be compared against.
6. **Faster and cleaner training.** Dynamic padding, `text_target` tokenization, gradient accumulation, warmup, a fixed seed and no hardcoded checkpoint path.

#### **Install dependencies**

In [1]:
!pip install -q transformers datasets evaluate rouge_score accelerate nltk huggingface_hub pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


#### **Imports, seed and configuration**
All tunable values live in one cell. Increase `TRAIN_SIZE` if your Colab session allows it, since more data is the most reliable way to raise ROUGE.

In [20]:
import gc
import json
import math
import random
import re

import evaluate
import nltk
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
MODEL_NAME = "facebook/bart-base"
TRAIN_SIZE = 16000
VAL_SIZE = 800
TEST_SIZE = 800
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
WARMUP_FRACTION = 0.05
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 142
SWEEP_SIZE = 300
N_BOOTSTRAP = 2000
OUTPUT_DIR = "./bart-news-summarizer"
FINAL_DIR = "./bart-summarizer-best"
HF_REPO_ID = "Jayywestty/bart-summarizer-epoch2"
PUSH_TO_HUB = False

FIXED_GEN_KWARGS = dict(num_beams=4, no_repeat_ngram_size=3, early_stopping=True)
SWEEP_MAX_LENGTHS = [64, 80, 100]
SWEEP_LENGTH_PENALTIES = [0.8, 1.2, 2.0]
OLD_GEN_KWARGS = dict(
    num_beams=4,
    max_length=MAX_TARGET_LENGTH,
    min_length=56,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [21]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

#### **Load tokenizer and model**
BART-base gives better summaries than T5-small at a size that still trains on a free Colab GPU.

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"Model loaded. Parameters: {model.num_parameters():,}")
print(f"Default generation max_length: {model.generation_config.max_length}")
clear_memory()

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Model loaded. Parameters: 139,420,416
Default generation max_length: None


#### **Load dataset**
CNN/DailyMail 3.0.0, reduced to fit Colab limits. The test split is loaded now but is not used until the final evaluation.

In [23]:
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

train_dataset = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_dataset = dataset["validation"].select(range(VAL_SIZE))
test_dataset = dataset["test"].select(range(TEST_SIZE))

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print("\nSample article:\n", train_dataset[0]["article"][:300])
print("\nSample summary:\n", train_dataset[0]["highlights"])

Train: 16000, Val: 800, Test: 800

Sample article:
 By . Anthony Bond . PUBLISHED: . 07:03 EST, 2 March 2013 . | . UPDATED: . 08:07 EST, 2 March 2013 . Three members of the same family who died in a static caravan from carbon monoxide poisoning would have been unconscious 'within minutes', investigators said today. The bodies of married couple John a

Sample summary:
 John and .
Audrey Cook were discovered alongside their daughter, Maureen .
They were found at Tremarle Home Park in Cornwall .
Investigators say the three died of carbon monoxide .
poisoning .


#### **Preprocessing**
Inputs are truncated but not padded here. `DataCollatorForSeq2Seq` pads each batch to its own longest sequence and sets label padding to -100, which is faster than padding everything to 1024 tokens.

In [24]:
def preprocess(batch):
    model_inputs = tokenizer(
        batch["article"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["highlights"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_tokenized = train_dataset.map(
    preprocess, batched=True, remove_columns=train_dataset.column_names
)
val_tokenized = val_dataset.map(
    preprocess, batched=True, remove_columns=val_dataset.column_names
)

print(train_tokenized)
print("Pad token id:", tokenizer.pad_token_id, "| Vocab size:", tokenizer.vocab_size)
clear_memory()

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 16000
})
Pad token id: 1 | Vocab size: 50265


#### **Metrics and helpers**
`rouge_scores` splits every text into sentences joined by newlines. This is what makes `rougeLsum` a real summary-level score. The same function is used during training, for the baseline and for the final evaluation, so all numbers are directly comparable.

In [25]:
rouge = evaluate.load("rouge")


def to_sentence_lines(text):
    return "\n".join(nltk.sent_tokenize(text.strip()))


def rouge_scores(predictions, references):
    preds = [to_sentence_lines(p) for p in predictions]
    refs = [to_sentence_lines(r) for r in references]
    result = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge_scores(decoded_preds, decoded_labels)
    result["gen_len"] = round(
        float(np.mean([np.count_nonzero(p != tokenizer.pad_token_id) for p in predictions])), 2
    )
    return result


def generate_summaries(model, tokenizer, texts, batch_size=16, **gen_kwargs):
    model.eval()
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    outputs = [None] * len(texts)
    use_cuda = torch.cuda.is_available()

    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        batch = [texts[i] for i in idx]
        enc = tokenizer(
            batch,
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_cuda):
            ids = model.generate(**enc, **gen_kwargs)
        for i, text in zip(idx, tokenizer.batch_decode(ids, skip_special_tokens=True)):
            outputs[i] = text.strip()
    return outputs


def lead_k(article, k=3):
    return " ".join(nltk.sent_tokenize(article.strip())[:k])

#### **Baseline: Lead-3**
Lead-3 takes the first three sentences of the article as the summary. It is a strong baseline on news data, so the fine-tuned model should beat it on ROUGE-1 and ROUGE-2 to justify the training. It is scored on the validation split here and on the test split at the end.

In [26]:
val_articles = val_dataset["article"]
val_references = val_dataset["highlights"]

BOILERPLATE = re.compile(
    r"PUBLISHED:|UPDATED:|Last updated|^By \.|^\W*$|\b\d{1,2}:\d{2}\s+(?:EST|GMT|BST|EDT)\b"
)
DATELINE = re.compile(
    r"^\s*(?:[A-Z][A-Za-z .,'\-]{0,60}\s)?\((?:CNN|Reuters|AP)\)\s*(?:--)?\s*"
)


def avg_words(texts):
    return round(float(np.mean([len(t.split()) for t in texts])), 1)


def clean_sentences(article):
    cleaned = []
    for sentence in nltk.sent_tokenize(article.strip()):
        sentence = DATELINE.sub("", sentence, count=1).strip()
        if BOILERPLATE.search(sentence) or len(sentence.split()) < 4:
            continue
        cleaned.append(sentence)
    return cleaned


def lead_original(article):
    return lead_k(article)


def lead_clean(article):
    return " ".join(clean_sentences(article)[:3]) or lead_k(article)


def lead_regex(article):
    return " ".join(re.split(r"(?<=[.!?])\s+", article.strip())[:3])


LEAD_VARIANTS = {
    "original": lead_original,
    "boilerplate_stripped": lead_clean,
    "regex_split": lead_regex,
}

lead_rows = []
for name, fn in LEAD_VARIANTS.items():
    preds = [fn(a) for a in val_articles]
    lead_rows.append({"variant": name, **rouge_scores(preds, val_references), "avg_words": avg_words(preds)})

lead_table = pd.DataFrame(lead_rows).sort_values("rougeLsum", ascending=False).reset_index(drop=True)
display(lead_table)
print("Reference average words:", avg_words(val_references))

LEAD_NAME = lead_table.loc[0, "variant"]
LEAD_FN = LEAD_VARIANTS[LEAD_NAME]
lead3_val = rouge_scores([LEAD_FN(a) for a in val_articles], val_references)
print("Selected Lead-3 variant:", LEAD_NAME)

for i in range(2):
    print("\nORIGINAL :", lead_original(val_articles[i])[:300])
    print("SELECTED :", LEAD_FN(val_articles[i])[:300])
    print("REFERENCE:", val_references[i])

,variant,rouge1,rouge2,rougeL,rougeLsum,avg_words
0,boilerplate_stripped,29.33,10.97,19.49,26.10,68.9
1,original,28.95,10.76,19.23,25.72,67.6
2,regex_split,28.83,10.73,19.29,25.64,67.4


Reference average words: 34.1
Selected Lead-3 variant: boilerplate_stripped

ORIGINAL : (CNN)Share, and your gift will be multiplied. That may sound like an esoteric adage, but when Zully Broussard selflessly decided to give one of her kidneys to a stranger, her generosity paired up with big data. It resulted in six patients receiving transplants.
SELECTED : Share, and your gift will be multiplied. That may sound like an esoteric adage, but when Zully Broussard selflessly decided to give one of her kidneys to a stranger, her generosity paired up with big data. It resulted in six patients receiving transplants.
REFERENCE: Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

ORIGINAL : (CNN)On the 6th of April 1996, San Jose Clash and DC United strode out in front of 31,683 expectant fans at the Spartan Stadium in San Jose, California. The historic occasion was the first ever Major League Soccer ma

#### **Training configuration**
Key differences from the first version:
- `generation_max_length=142` and `generation_num_beams=2` so evaluation during training produces full-length summaries. Beam search is kept at 2 here for speed, and the final evaluation uses 4.
- `load_best_model_at_end=True` with `metric_for_best_model="rougeLsum"` picks the checkpoint automatically.
- Effective batch size is 16 (4 per device, 4 accumulation steps), which gives more stable updates than a batch of 2.
- Warmup and label smoothing are added for steadier training.

In [27]:
steps_per_epoch = math.ceil(len(train_tokenized) / (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS))
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_FRACTION)
print(f"Steps per epoch: {steps_per_epoch}, total: {total_steps}, warmup: {warmup_steps}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    label_smoothing_factor=0.1,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=2,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeLsum",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Steps per epoch: 1000, total: 5000, warmup: 250


#### **Train**

In [28]:
clear_memory()
train_result = trainer.train()
print(train_result.metrics)

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,3.441067,3.330971,31.770000,12.190000,22.430000,29.090000,59.510000
2,3.196764,3.303604,31.280000,11.870000,21.820000,28.670000,68.860000
3,3.105544,3.276253,31.880000,12.340000,22.210000,29.100000,70.200000
4,2.988098,3.266292,32.610000,12.690000,23.140000,29.940000,63.600000
5,2.927168,3.270099,33.050000,13.240000,23.520000,30.470000,67.050000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


{'train_runtime': 4986.3702, 'train_samples_per_second': 16.044, 'train_steps_per_second': 1.003, 'total_flos': 4.693227739619328e+16, 'train_loss': 3.1804022552490236, 'epoch': 5.0}


#### **Per-epoch results and selected checkpoint**
This table replaces the manual choice of epoch 2. The trainer has already reloaded the best checkpoint into `trainer.model`.

In [29]:
history = pd.DataFrame(trainer.state.log_history)
epoch_eval = history[history["eval_loss"].notna()].copy() if "eval_loss" in history else history
cols = [c for c in ["epoch", "step", "eval_loss", "eval_rouge1", "eval_rouge2", "eval_rougeL", "eval_rougeLsum", "eval_gen_len"] if c in epoch_eval.columns]
display(epoch_eval[cols].reset_index(drop=True))

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best rougeLsum on validation:", trainer.state.best_metric)

,epoch,step,eval_loss,eval_rouge1,eval_rouge2,eval_rougeL,eval_rougeLsum,eval_gen_len
0,1.0,1000,3.330971,31.77,12.19,22.43,29.09,59.51
1,2.0,2000,3.303604,31.28,11.87,21.82,28.67,68.86
2,3.0,3000,3.276253,31.88,12.34,22.21,29.10,70.20
3,4.0,4000,3.266292,32.61,12.69,23.14,29.94,63.60
4,5.0,5000,3.270099,33.05,13.24,23.52,30.47,67.05


Best checkpoint: ./bart-news-summarizer/checkpoint-5000
Best rougeLsum on validation: 30.47


In [30]:
best_model = trainer.model
best_model.eval()

sweep_articles = val_articles[:SWEEP_SIZE]
sweep_references = val_references[:SWEEP_SIZE]

configs = [("old_settings", OLD_GEN_KWARGS)]
for max_len in SWEEP_MAX_LENGTHS:
    for penalty in SWEEP_LENGTH_PENALTIES:
        configs.append(
            (
                f"max{max_len}_lp{penalty}",
                dict(**FIXED_GEN_KWARGS, max_length=max_len, min_length=0, length_penalty=penalty),
            )
        )

sweep_rows = []
for name, kwargs in configs:
    preds = generate_summaries(best_model, tokenizer, sweep_articles, **kwargs)
    scores = rouge_scores(preds, sweep_references)
    sweep_rows.append({"config": name, **scores, "avg_words": avg_words(preds)})
    print(name, "rougeLsum:", scores["rougeLsum"], flush=True)
    clear_memory()

sweep_table = pd.DataFrame(sweep_rows).sort_values("rougeLsum", ascending=False).reset_index(drop=True)
display(sweep_table)
print("Reference average words:", avg_words(sweep_references))

BEST_CONFIG_NAME = sweep_table.loc[0, "config"]
BEST_GEN_KWARGS = dict(configs)[BEST_CONFIG_NAME]
FINAL_GEN_KWARGS = BEST_GEN_KWARGS
print("Selected decoding:", BEST_CONFIG_NAME, BEST_GEN_KWARGS)

old_settings rougeLsum: 30.51
max64_lp0.8 rougeLsum: 31.57
max64_lp1.2 rougeLsum: 31.53
max64_lp2.0 rougeLsum: 31.58
max80_lp0.8 rougeLsum: 31.16
max80_lp1.2 rougeLsum: 31.15
max80_lp2.0 rougeLsum: 31.02
max100_lp0.8 rougeLsum: 31.1
max100_lp1.2 rougeLsum: 31.08
max100_lp2.0 rougeLsum: 30.92


,config,rouge1,rouge2,rougeL,rougeLsum,avg_words
0,max64_lp2.0,34.25,14.41,24.93,31.58,46.7
1,max64_lp0.8,34.16,14.48,25.00,31.57,44.8
2,max64_lp1.2,34.10,14.30,24.87,31.53,45.9
3,max80_lp0.8,33.92,14.41,24.58,31.16,48.3
4,max80_lp1.2,33.78,14.23,24.37,31.15,50.8
5,max100_lp0.8,33.75,14.38,24.46,31.10,49.0
6,max100_lp1.2,33.65,14.25,24.22,31.08,52.6
7,max80_lp2.0,33.76,14.19,24.40,31.02,52.7
8,max100_lp2.0,33.60,14.20,24.20,30.92,55.0
9,old_settings,33.12,14.05,23.60,30.51,59.4


Reference average words: 33.8
Selected decoding: max64_lp2.0 {'num_beams': 4, 'no_repeat_ngram_size': 3, 'early_stopping': True, 'max_length': 64, 'min_length': 0, 'length_penalty': 2.0}


#### **Final evaluation with full decoding settings**
The best model is scored on validation and then, once, on the held-out test split. Both use beam search 4, a minimum length of 56 and no repeated trigrams. Lead-3 is scored on the same test articles.

In [31]:
best_model = trainer.model
best_model.eval()

test_articles = test_dataset["article"]
test_references = test_dataset["highlights"]

val_preds = generate_summaries(best_model, tokenizer, val_articles, **BEST_GEN_KWARGS)
model_val = rouge_scores(val_preds, val_references)

test_preds = generate_summaries(best_model, tokenizer, test_articles, **BEST_GEN_KWARGS)
model_test = rouge_scores(test_preds, test_references)

lead_val_preds = [LEAD_FN(a) for a in val_articles]
lead_test_preds = [LEAD_FN(a) for a in test_articles]
lead3_val = rouge_scores(lead_val_preds, val_references)
lead3_test = rouge_scores(lead_test_preds, test_references)

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(best_model.device)
base_val_preds = generate_summaries(base_model, tokenizer, val_articles, **BEST_GEN_KWARGS)
base_test_preds = generate_summaries(base_model, tokenizer, test_articles, **BEST_GEN_KWARGS)
untrained_val = rouge_scores(base_val_preds, val_references)
untrained_test = rouge_scores(base_test_preds, test_references)
del base_model
clear_memory()

results = pd.DataFrame(
    {
        "Untrained BART-base (val)": untrained_val,
        "Lead-3 (val)": lead3_val,
        "Fine-tuned (val)": model_val,
        "Untrained BART-base (test)": untrained_test,
        "Lead-3 (test)": lead3_test,
        "Fine-tuned (test)": model_test,
    }
).T
display(results)

lengths = pd.Series(
    {
        "Reference (test)": avg_words(test_references),
        "Fine-tuned (test)": avg_words(test_preds),
        "Lead-3 (test)": avg_words(lead_test_preds),
        "Untrained (test)": avg_words(base_test_preds),
    },
    name="avg_words",
)
display(lengths)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

,rouge1,rouge2,rougeL,rougeLsum
Untrained BART-base (val),30.59,10.72,20.33,26.89
Lead-3 (val),29.33,10.97,19.49,26.10
Fine-tuned (val),33.14,13.27,23.69,30.42
Untrained BART-base (test),31.42,11.85,21.31,27.78
Lead-3 (test),30.15,11.79,20.46,26.87
Fine-tuned (test),32.82,12.96,23.33,30.11


,avg_words
Reference (test),34.3
Fine-tuned (test),46.7
Lead-3 (test),66.8
Untrained (test),46.9


In [32]:
def per_example_rouge(predictions, references):
    preds = [to_sentence_lines(p) for p in predictions]
    refs = [to_sentence_lines(r) for r in references]
    result = rouge.compute(
        predictions=preds, references=refs, use_stemmer=True, use_aggregator=False
    )
    return {k: np.array(v) * 100 for k, v in result.items()}


def paired_bootstrap(scores_a, scores_b, n=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    diff = scores_a - scores_b
    idx = rng.integers(0, len(diff), size=(n, len(diff)))
    means = diff[idx].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return float(diff.mean()), float(low), float(high), float((means <= 0).mean())


def single_bootstrap(scores, n=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(scores), size=(n, len(scores)))
    means = scores[idx].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return float(scores.mean()), float(low), float(high)


model_pe = per_example_rouge(test_preds, test_references)
lead_pe = per_example_rouge(lead_test_preds, test_references)
base_pe = per_example_rouge(base_test_preds, test_references)

comparison_rows = []
for label, other in [("fine-tuned vs lead-3", lead_pe), ("fine-tuned vs untrained", base_pe)]:
    for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
        diff, low, high, share = paired_bootstrap(model_pe[metric], other[metric])
        comparison_rows.append(
            {
                "comparison": label,
                "metric": metric,
                "mean_gain": round(diff, 2),
                "ci95_low": round(low, 2),
                "ci95_high": round(high, 2),
                "share_not_better": round(share, 4),
            }
        )
bootstrap_table = pd.DataFrame(comparison_rows)
display(bootstrap_table)

interval_rows = []
for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
    mean, low, high = single_bootstrap(model_pe[metric])
    interval_rows.append(
        {"metric": metric, "test_mean": round(mean, 2), "ci95_low": round(low, 2), "ci95_high": round(high, 2)}
    )
model_interval_table = pd.DataFrame(interval_rows)
display(model_interval_table)

,comparison,metric,mean_gain,ci95_low,ci95_high,share_not_better
0,fine-tuned vs lead-3,rouge1,2.66,1.86,3.48,0.0000
1,fine-tuned vs lead-3,rouge2,1.18,0.48,1.88,0.0000
2,fine-tuned vs lead-3,rougeL,2.85,2.16,3.56,0.0000
3,fine-tuned vs lead-3,rougeLsum,3.24,2.48,4.03,0.0000
4,fine-tuned vs untrained,rouge1,1.37,0.54,2.22,0.0005
5,fine-tuned vs untrained,rouge2,1.12,0.37,1.87,0.0005
6,fine-tuned vs untrained,rougeL,2.00,1.28,2.75,0.0000
7,fine-tuned vs untrained,rougeLsum,2.33,1.51,3.15,0.0000


,metric,test_mean,ci95_low,ci95_high
0,rouge1,32.81,31.98,33.67
1,rouge2,12.97,12.20,13.77
2,rougeL,23.33,22.58,24.09
3,rougeLsum,30.12,29.34,30.93


#### **Qualitative check**
A few test articles with the reference and the generated summary, since ROUGE alone does not show fluency or factual errors.

In [33]:
for i in random.Random(SEED).sample(range(len(test_dataset)), 3):
    print("ARTICLE (start):", test_articles[i][:300].replace("\n", " "))
    print("REFERENCE      :", test_references[i])
    print("MODEL          :", test_preds[i])
    print()

ARTICLE (start): (CNN)For the fifth year in a row in 2014, ambush attacks on police officers were the No. 1 cause of felonious deaths of law enforcement officers in the line of duty. Nevertheless, Google continues to market a smartphone application that lets lawbreakers pinpoint the location of police officers in th
REFERENCE      : David A. Clarke Jr. and Jonathan Thompson: Why does Google have an app that ambushes police?
With Waze, we are confronted with a tool that can be lethal to police officers and deputies .
MODEL          : Google's popular real-time traffic app, Waze, uses GPS navigation and crowdsourcing to alert users to traffic jams, automobile accidents, stalled cars .
Waze's "traffic cop" feature poses an enormous risk to deputies and police officers .
In 2013, 10,076 people were

ARTICLE (start): (CNN)I remember the day I stopped praying. It was the day after my little brother, Jimmy, died of cancer. He was 25. I was so angry at God. I was 27 at the time, and, like most

#### **Try the model on a new article**

In [34]:
def normalize_text(text):
    return " ".join(text.split())


def summarize(text, model=best_model, tokenizer=tokenizer, **overrides):
    kwargs = {**FINAL_GEN_KWARGS, **overrides}
    return generate_summaries(model, tokenizer, [normalize_text(text)], batch_size=1, **kwargs)[0]

In [35]:
ARTICLE = """
The Vice Chancellor, Federal University of Technology and Environmental Sciences, Iyin Ekiti, Prof. Gbenga Aribisala, has said that the new institution will begin the admission process in September.

Aribisala said that the admission process would follow the National Universities Commission Resource Verification exercise taking place soon in the university.

The VC, who spoke in Ado Ekiti on Sunday at a reception to celebrate the 90th birthday of his mother, Deaconess Felicia Aribisala, also canvassed support from well-meaning Nigerians to the institution, saying, “A technology-based institution of this nature is capital-intensive”.

He said, “The NUC is coming for Resource Verification of all the 36 programmes that we are trying to offer. As soon as they come, by the special grace of God, we have provided those things that will be needed.
“We have provided a modern laboratory for all the programmes. We have a library now. We have classrooms fixed.

“We have offices and furniture fixed. We have all of those things. So we are very confident we are going to scale through.

“By the time we now scale through, by the special grace of God, by September this year, we are going to ask those who are interested in our university to do Change of University, and admission will begin. That is the icing. And after that, recruitment of staff will just follow”.

The VC, who said that funding of education should not be left to the government alone, said, “Universities need a lot of funding. Funding is a major challenge. You have to provide facilities and all of those things.
“So, as I speak to you, we (FUTES) do not have enough funds. That’s why we keep appealing and going to people because the government cannot do it all alone. We have been visiting some people who are public-spirited, people who like education, tertiary education.

“If we have people who want to donate buildings, we are going to name such after them; people who want to give scholarships; people who want to build hostels in such a manner that it is their own and they will take rent and all of those things.

“I think the funding is crucial because if you look at the nature of our university, University of Technology and Environmental Sciences, it is capital-intensive, it is technology-based.

“It means we need a lot of equipment. As I said, the government cannot do everything. So we need help at this time financially,” the VC said.

Aribisala disclosed that the land issue, which could have been a challenge to the university, had been resolved amicably with an agreement made with the concerned families.

“As I speak to you now, it has been resolved. The 200 hectares that have been donated to the university are very intact.

“There has been an agreement. The community and government will also pay some compensation to the families.

“So they are now at peace. The community is not trying to force the land. I think that was the kind of misconception that happened at the time,” the Vice Chancellor said.
"""

print(summarize(ARTICLE))

The Vice Chancellor of the Federal University of Technology and Environmental Sciences, Iyin Ekiti, Prof. Gbenga Aribisala, has said that the admission process will begin in September .
The VC said that funding of education should not be left to the government alone .
Arib


#### **Save the final model and results**
The best checkpoint is saved to a clean folder, reloaded to confirm it works, and the scores are written to `results.json`.

In [36]:
best_model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

reloaded_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_DIR).to(best_model.device)
reloaded_tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
check = summarize(ARTICLE, model=reloaded_model, tokenizer=reloaded_tokenizer)
assert len(check.split()) > 10
print("Reloaded model works. Saved at", FINAL_DIR)

with open("results.json", "w") as f:
    json.dump(
        {
            "train_size": TRAIN_SIZE,
            "epochs": NUM_EPOCHS,
            "best_checkpoint": trainer.state.best_model_checkpoint,
            "validation_best_rougeLsum": trainer.state.best_metric,
            "lead3_variant": LEAD_NAME,
            "decoding_config": BEST_CONFIG_NAME,
            "decoding_kwargs": BEST_GEN_KWARGS,
            "untrained_val": untrained_val,
            "lead3_val": lead3_val,
            "model_val": model_val,
            "untrained_test": untrained_test,
            "lead3_test": lead3_test,
            "model_test": model_test,
            "bootstrap": bootstrap_table.to_dict(orient="records"),
            "model_test_intervals": model_interval_table.to_dict(orient="records"),
            "decoding_sweep": sweep_table.to_dict(orient="records"),
            "lead3_variants": lead_table.to_dict(orient="records"),
        },
        f,
        indent=2,
    )

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Reloaded model works. Saved at ./bart-summarizer-best


#### **Optional: upload to the Hugging Face Hub**
Set `PUSH_TO_HUB = True` in the configuration cell to run this. The repo id is `HF_REPO_ID`. If that repo already holds the earlier epoch 2 model, this will overwrite it, so change the id if you want to keep both.

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import notebook_login

    notebook_login()
    reloaded_model.push_to_hub(HF_REPO_ID)
    reloaded_tokenizer.push_to_hub(HF_REPO_ID)
    print(f"Model uploaded to https://huggingface.co/{HF_REPO_ID}")
else:
    print("PUSH_TO_HUB is False, skipping upload.")